# Notebook 06 — Multimodal CLIP Retrieval Demo

**Tri-Modal Retrieval Architecture:**

| Layer | Model | Collection | Dim |
|-------|-------|-----------|-----|
| Text RAG | all-MiniLM-L6-v2 | kapruka_catalog | 384 |
| CLIP Image | clip-vit-base-patch32 | kapruka_clip_images | 512 |
| Fusion | Weighted 60/40 | — | — |

**Key concept:** CLIP encodes text and images into the *same* 512-dim space.  
A text query `"red velvet cake"` and an image of a red velvet cake will have high cosine similarity — without any text label on the image.

> **Memory note:** CLIP ViT-B/32 loads ~600 MB on CPU. Run cells top-to-bottom.  
> Each cell calls `gc.collect()` to manage peak RAM.

In [1]:
# ── Cell 1: Setup & component loading ────────────────────────────────────────
import sys, gc
from pathlib import Path

import torch
torch.set_num_threads(2)          # Cap CPU parallelism — prevents OOM on low-RAM machines

sys.path.insert(0, str(Path('..').resolve()))

from config.settings import Settings
from src.multimodal.clip_encoder import CLIPEncoder
from src.multimodal.image_store import ImageVectorStore
from src.multimodal.fusion_ranker import FusionRanker
from src.memory.long_term import CatalogVectorStore
# NOTE: VisualSearchAgent is imported lazily in Cell 7 to avoid double-loading CLIP

settings  = Settings()
encoder   = CLIPEncoder()          # Loads CLIP once (~90s on CPU) — singleton, reused by all cells
img_store = ImageVectorStore()
txt_store = CatalogVectorStore(settings)
fusion    = FusionRanker()

info = img_store.get_collection_info()
print(f'CLIP collection : {info["name"]}')
print(f'Image vectors   : {info["vectors_count"]}')
print(f'Vector dim      : {info["vector_size"]} (CLIP shared space)')
print(f'Distance metric : {info["distance"]}')

d:\Zuu Crew Agentic AI\Projects\Mini Project 03\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-14 12:53:35.672 | INFO     | src.multimodal.clip_encoder:__init__:57 - Loading CLIP model: openai/clip-vit-base-patch32
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5500.67it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-14 12:57:54.834 | INFO     | src.multimodal.clip_encoder:__init__:64 - CLIP model loaded


📦 Loading embedding model: all-MiniLM-L6-v2…


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2393.37it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready
CLIP collection : kapruka_clip_images
Image vectors   : 215
Vector dim      : 512 (CLIP shared space)
Distance metric : Cosine


In [2]:
# ── Cell 2: Pure CLIP cross-modal retrieval ───────────────────────────────────
# Text query → image collection search
# No text labels on images — match is purely visual embedding similarity

query = 'red velvet birthday cake'
print(f'Query : "{query}"')
print('='*60)

results = img_store.search(query, top_k=5)

print(f'  {"#":<4} {"Product":<40} {"Category":<14} {"Price LKR":>10} {"CLIP Score":>11}')
print(f'  {"-"*82}')
for i, r in enumerate(results, 1):
    p = r['product']
    print(f"  {i:<4} {p.get('product_name','?')[:38]:<40} "
          f"{p.get('category','?'):<14} "
          f"{p.get('price_lkr',0):>10,.0f} "
          f"{r['clip_score']:>11.4f}")

Query : "red velvet birthday cake"
  #    Product                                  Category        Price LKR  CLIP Score
  ----------------------------------------------------------------------------------
  1    Princess Unicorn Birthday Cake 1.5kg     cakes               8,500      0.3020
  2    Carrot Walnut Cake 1.5kg                 cakes               7,200      0.2933
  3    Mango Fresh Cream Cake 1.5kg             cakes               6,800      0.2664
  4    Fruit Cake 0.5kg - Traditional           cakes               3,500      0.2662
  5    Number Cake 0.5kg - Nut-Free Custom      cakes               6,000      0.2633


In [3]:
# ── Cell 3: Display actual product images retrieved by CLIP ──────────────────
# Each card shows a UNIQUE product image (deduplicated by image_url).
# Images are served from Kapruka's static2 CDN.
# Results are ranked by CLIP cosine similarity — highest visual match first.
from IPython.display import display, HTML

html_cards = ""
for i, r in enumerate(results[:3], 1):
    p        = r['product']
    name     = p.get('product_name', '?')
    price    = p.get('price_lkr', 0)
    category = p.get('category', '?')
    prod_url = p.get('product_url', p.get('url', '#'))
    img_url  = p.get('image_url', '')
    allergens = ', '.join(p.get('contains_allergens', [])) or 'None'
    score    = r['clip_score']

    html_cards += f"""
    <div style="display:inline-block;margin:10px;vertical-align:top;width:230px;
                font-family:Arial,sans-serif;border:1px solid #ddd;border-radius:8px;
                overflow:hidden;box-shadow:0 2px 6px rgba(0,0,0,.08)">
      <a href="{prod_url}" target="_blank">
        <img src="{img_url}" width="230" height="180"
             style="display:block;object-fit:cover"
             onerror="this.src='';this.style.background='#f5f5f5';this.style.height='180px'">
      </a>
      <div style="padding:10px">
        <div style="font-weight:bold;font-size:13px;margin-bottom:4px">{name}</div>
        <div style="color:#e74c3c;font-weight:bold">LKR {price:,.0f}</div>
        <div style="color:#888;font-size:11px;margin-top:2px">{category}</div>
        <div style="margin-top:6px;padding:4px 6px;background:#f0f7ff;border-radius:4px;font-size:11px">
          CLIP score: <b>{score:.4f}</b>
        </div>
        <div style="font-size:11px;color:#c0392b;margin-top:4px">
          Allergens: {allergens}
        </div>
        <a href="{prod_url}" target="_blank"
           style="display:block;margin-top:8px;font-size:11px;color:#2980b9">
          View on kapruka.com →
        </a>
      </div>
    </div>"""

display(HTML(f'<div style="margin:10px 0"><b>Top 3 unique visual matches for:</b> "{query}"</div>' + html_cards))

In [4]:
# ── Cell 4: Fusion — Text RAG + CLIP combined ─────────────────────────────────
# Shows how fusion rescores products from both retrieval layers

fquery = 'birthday gift for mother'
print(f'Fusion query: "{fquery}"')
print('='*60)

text_results  = txt_store.search(fquery, top_k=8)
image_results = img_store.search(fquery, top_k=8)
fused         = fusion.fuse(text_results, image_results, top_k=8)

print(f'Text RAG returned  : {len(text_results)} products')
print(f'CLIP returned      : {len(image_results)} products')
print(f'After fusion (top8): {len(fused)} products')
print()
print(f'  {"#":<3} {"Product":<38} {"Fused":>6} {"Text":>6} {"Image":>6} {"Source":<12}')
print(f'  {"-"*78}')
for i, r in enumerate(fused, 1):
    name = r['product'].get('product_name', '?')[:36]
    print(f"  {i:<3} {name:<38} "
          f"{r['fused_score']:>6.3f} "
          f"{r['text_score']:>6.3f} "
          f"{r['image_score']:>6.3f} "
          f"{r['retrieval_source']:<12}")

Fusion query: "birthday gift for mother"


2026-04-14 12:58:03.983 | DEBUG    | src.multimodal.fusion_ranker:fuse:127 - Fusion: 8 text + 8 image -> 16 merged -> top 8


Text RAG returned  : 8 products
CLIP returned      : 8 products
After fusion (top8): 8 products

  #   Product                                 Fused   Text  Image Source      
  ------------------------------------------------------------------------------
  1   Mother's Day Flowers Card — Luxury H    0.600  1.000  0.000 text_only   
  2   Black Forest Cake 0.5kg                 0.400  0.000  1.000 image_only  
  3   Hazelnut Crunch Chocolate Cake 1.5kg    0.313  0.000  0.781 image_only  
  4   Mother's Day Flowers Card — Mini Pos    0.271  0.452  0.000 text_only   
  5   Mother's Day Flowers Card — 3D Pop-U    0.204  0.340  0.000 text_only   
  6   Number Cake 1kg - Nut-Free Custom       0.174  0.000  0.436 image_only  
  7   Mother's Day Flowers Card — Large A4    0.172  0.286  0.000 text_only   
  8   Baby's Breath Bouquet - 24 Stems        0.150  0.000  0.375 image_only  


In [5]:
# ── Cell 5: CLIP-exclusive hits — products text search MISSED ─────────────────
# These are products the text search missed but CLIP retrieved visually

image_only = [r for r in fused if r['retrieval_source'] == 'image_only']
both       = [r for r in fused if r['retrieval_source'] == 'both']
text_only  = [r for r in fused if r['retrieval_source'] == 'text_only']

print(f'Retrieved by BOTH text + image : {len(both)}')
print(f'Retrieved by TEXT only         : {len(text_only)}')
print(f'Retrieved by IMAGE (CLIP) only : {len(image_only)}')

if image_only:
    print('\nCLIP-exclusive hits (text search missed these):')
    for r in image_only:
        p = r['product']
        print(f"  {p.get('product_name','?')} "
              f"| {p.get('category')} "
              f"| LKR {p.get('price_lkr',0):,.0f} "
              f"| visual={r['image_score']:.3f}")
else:
    print('\n(No exclusive CLIP hits for this query — try a more visual query in Cell 6)')

gc.collect()

Retrieved by BOTH text + image : 0
Retrieved by TEXT only         : 4
Retrieved by IMAGE (CLIP) only : 4

CLIP-exclusive hits (text search missed these):
  Black Forest Cake 0.5kg | cakes | LKR 4,300 | visual=1.000
  Hazelnut Crunch Chocolate Cake 1.5kg | cakes | LKR 8,300 | visual=0.781
  Number Cake 1kg - Nut-Free Custom | cakes | LKR 6,600 | visual=0.436
  Baby's Breath Bouquet - 24 Stems | flowers | LKR 3,800 | visual=0.375


221

In [6]:
# ── Cell 6: 5 diverse queries across all CLIP categories ──────────────────────
# Shows CLIP works across cakes, flowers, chocolates, hampers

QUERIES = [
    'chocolate birthday cake with candles',
    'elegant flower bouquet for anniversary',
    'luxury gift hamper with multiple items',
    'something colourful and festive',
    'traditional Sri Lankan sweet treat',
]

print(f'  {"Query":<45} {"Top CLIP Match":<38} {"Cat":<12} {"Score":>6}')
print(f'  {"-"*106}')
for q in QUERIES:
    res = img_store.search(q, top_k=1)
    if res:
        p = res[0]['product']
        print(f"  {q[:43]:<45} "
              f"{p.get('product_name','?')[:36]:<38} "
              f"{p.get('category','?')[:10]:<12} "
              f"{res[0]['clip_score']:>6.3f}")

gc.collect()

  Query                                         Top CLIP Match                         Cat           Score
  ----------------------------------------------------------------------------------------------------------
  chocolate birthday cake with candles          Tres Leches Cake 1kg                   cakes         0.279
  elegant flower bouquet for anniversary        Baby's Breath Bouquet - 24 Stems       flowers       0.327
  luxury gift hamper with multiple items        Raffles Praline & Nut Box 100g         chocolates    0.346
  something colourful and festive               Purple Lavender Bundle - 12 Stems      flowers       0.235
  traditional Sri Lankan sweet treat            Number Cake 1.5kg - Nut-Free Custom    cakes         0.287


245

In [7]:
# ── Cell 7: VisualSearchAgent — full end-to-end multimodal recommendation ─────
# Lazy import here — VisualSearchAgent reuses the CLIPEncoder singleton already
# loaded in Cell 1, so no extra CLIP model is loaded into RAM.

from src.agents.visual_search_agent import VisualSearchAgent
from src.llm.client import LLMClient

llm    = LLMClient(settings)
visual = VisualSearchAgent(llm_client=llm, settings=settings)

query             = "I want a beautiful cake for my mum's 60th birthday"
recipient_context = 'Recipient: Mother, Age 60. Allergies: nuts. Preferences: flowers, tea, dark chocolate.'

print(f'Query     : "{query}"')
print(f'Recipient : {recipient_context}')
print('='*60)
print('Calling Text RAG + CLIP + Fusion + Claude...\n')

recommendation = visual.search(
    query=query,
    recipient_context=recipient_context,
    budget_max=8000,
    occasion='birthday',
    top_k=5,
)
print(recommendation)

gc.collect()

📦 Loading embedding model: all-MiniLM-L6-v2…


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12170.76it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-04-14 12:58:11.312 | INFO     | src.agents.visual_search_agent:search:69 - VisualSearchAgent | Query: 'I want a beautiful cake for my mum's 60th birthday'


✅ Embedding model ready
Query     : "I want a beautiful cake for my mum's 60th birthday"
Recipient : Recipient: Mother, Age 60. Allergies: nuts. Preferences: flowers, tea, dark chocolate.
Calling Text RAG + CLIP + Fusion + Claude...



2026-04-14 12:58:12.015 | DEBUG    | src.agents.visual_search_agent:search:73 -   Text RAG: 5 results
2026-04-14 12:58:12.575 | DEBUG    | src.agents.visual_search_agent:search:77 -   CLIP image: 5 results
2026-04-14 12:58:12.576 | DEBUG    | src.multimodal.fusion_ranker:fuse:127 - Fusion: 5 text + 5 image -> 10 merged -> top 5
2026-04-14 12:58:12.577 | DEBUG    | src.agents.visual_search_agent:search:85 -   Fused: 5 results


Dear customer,

For your mother's 60th birthday, I would recommend the following gifts:

1. Classic Chocolate Truffle Cake 1.5kg (LKR 7,000)
   This cake is a strong visual match, scoring a perfect 1.00 on image similarity. It features rich chocolate ganache with nutty praline layers, which aligns well with your mother's preferences for dark chocolate and no nut allergies. At 1.5kg, it's a generous size that can be shared with family and friends to celebrate this special occasion.

2. Classic Chocolate Truffle Cake 500g (LKR 3,500)
   This 500g version of the Classic Chocolate Truffle Cake is a great option if you're looking for a slightly smaller cake. It still has the same delicious chocolate and praline flavors that your mother will love. The 500g size is perfect for an intimate birthday celebration.

Both of these cakes are nut-free, which is important given your mother's allergy. They also feature the classic chocolate flavors that align with her preferences.

I hope these recomme

68

In [8]:
# ── Cell 8: Phase 8 Validation Checklist ──────────────────────────────────────
print('='*60)
print('MULTIMODAL VALIDATION — Phase 8')
print('='*60)

all_pass = True

# Check 1: images on disk
image_dir   = Path('../data/images')
image_count = len(list(image_dir.glob('*'))) if image_dir.exists() else 0
ok = image_count >= 100
print(f'[{"OK  " if ok else "FAIL"}] Images on disk          : {image_count}  (need >= 100)')
if not ok: all_pass = False

# Check 2: manifest
manifest = Path('../data/image_manifest.json')
ok = manifest.exists()
print(f'[{"OK  " if ok else "FAIL"}] image_manifest.json     : {"exists" if ok else "MISSING"}')
if not ok: all_pass = False

# Check 3: Qdrant CLIP collection
clip_info = img_store.get_collection_info()
n = clip_info['vectors_count']
ok = n >= 100
print(f'[{"OK  " if ok else "FAIL"}] CLIP vectors in Qdrant  : {n}  (need >= 100)')
if not ok: all_pass = False

# Check 4: cross-modal search
hits = img_store.search('birthday cake', top_k=3)
ok = len(hits) > 0
print(f'[{"OK  " if ok else "FAIL"}] Cross-modal search       : {len(hits)} results for "birthday cake"')
for r in hits:
    print(f'          -> {r["product"].get("product_name","?")[:45]}  '
          f'CLIP={r["clip_score"]:.4f}')
if not ok: all_pass = False

# Check 5: fusion
t_res = txt_store.search('birthday cake', top_k=5)
i_res = img_store.search('birthday cake', top_k=5)
f_res = fusion.fuse(t_res, i_res, top_k=5)
ok = len(f_res) > 0
print(f'[{"OK  " if ok else "FAIL"}] Fusion ranker            : {len(f_res)} fused results')
if f_res:
    top = f_res[0]
    print(f'          Top: {top["product"].get("product_name","?")[:40]}  '
          f'score={top["fused_score"]:.4f}  src={top["retrieval_source"]}')
if not ok: all_pass = False

print('='*60)
if all_pass:
    print('\n[ALL PASS] Tri-Modal RAG: Text + CLIP + Fusion OPERATIONAL')
else:
    print('\n[FAIL] One or more checks failed.')

MULTIMODAL VALIDATION — Phase 8
[FAIL] Images on disk          : 0  (need >= 100)
[FAIL] image_manifest.json     : MISSING
[OK  ] CLIP vectors in Qdrant  : 215  (need >= 100)
[OK  ] Cross-modal search       : 3 results for "birthday cake"
          -> Tiramisu Cake 0.5kg  CLIP=0.2848
          -> Dairy-Free Dark Chocolate Cake 0.5kg  CLIP=0.2842
          -> Hazelnut Crunch Chocolate Cake 1.5kg  CLIP=0.2838


2026-04-14 12:58:18.088 | DEBUG    | src.multimodal.fusion_ranker:fuse:127 - Fusion: 5 text + 5 image -> 10 merged -> top 5


[OK  ] Fusion ranker            : 5 fused results
          Top: Classic Chocolate Truffle Cake 2kg — Bir  score=0.6000  src=text_only

[FAIL] One or more checks failed.
